# 02 - IEEE 13-node unbalanced feeder

## Objective

Preserve single- and two-phase feeder topology and compare phase-specific voltage magnitudes from direct OpenDSS with the CEPT public CLI.

## Source, assumptions, and units

The source is the IEEE 13-node OpenDSS feeder bundled in the installed CEPT wheel. The comparison point is bus `671`; phase 1, 2, and 3 correspond to A, B, and C. The feeder's declared data are used unchanged. Voltage magnitude is line-to-neutral per unit (`pu`). This is a public demonstrator/reference feeder, not a user's physical project.

## Prediction

The three phase voltages at bus 671 will not be represented safely by one balanced number. The direct OpenDSS and CEPT values should agree within the teaching tolerance when both use the bundled source.

## Action

Solve the bundled source directly, then run `cept study demo unbalanced-load-flow` in a separate exact run directory. The CEPT command streams its output into the originating cell.

## Verification

Read the CEPT solver table from `results.json`, run `cept study verify` on that exact directory, and compare phase identities before comparing values.

## Interpretation

Phase-specific spread is an observation from this solver run. Agreement between two routes through the same public source is a workflow regression check, not independent validation.

## Exercise

Change `BUS_TO_INSPECT` to another named bus present in the direct feeder, inspect its three phase values, and state whether the phase spread increased or decreased. Rerun from a restarted kernel.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
#@title Setup — run once, then read the results below
import contextlib, io, urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
_helper_output = io.StringIO()
with contextlib.redirect_stdout(_helper_output):
    exec(compile(_blob, "lesson helper", "exec"))
for _line in _helper_output.getvalue().splitlines():
    if "lesson helpers ready" not in _line.lower():
        print(_line)
print("Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.")


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


### Direct reference - solve without CEPT

The bundled IEEE13 feeder is loaded straight into OpenDSS. `Solve` must report converged first.


In [2]:
#@title Under the hood - direct OpenDSS solve (optional)
MASTER_DSS = ieee13_master()
BUS_TO_INSPECT = '671'
import opendssdirect as dss

dss.Basic.ClearAll()
dss.Basic.DataPath(str(MASTER_DSS.parent))
dss.Text.Command(f'Redirect "{MASTER_DSS}"')
dss.Text.Command('CalcVoltageBases')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
os.chdir(WORKSPACE)
print("Direct OpenDSS solve finished: the IEEE 13-node feeder converged.")


Direct OpenDSS solve finished: the IEEE 13-node feeder converged.


### Read bus 671 back

Three-phase voltage magnitude, solver-returned. Unbalanced laterals are the point of this feeder.


In [3]:
#@title Under the hood — direct OpenDSS readback (optional)
dss.Circuit.SetActiveBus(BUS_TO_INSPECT)
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
table(['source', 'bus', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', BUS_TO_INSPECT, phase, value, 'pu') for phase, value in direct_by_phase.items()])
assert all(value > 0 for value in direct_by_phase.values())

| source | bus | phase | voltage magnitude | unit |
| --- | --- | --- | --- | --- |
| direct OpenDSS | 671 | 1 | 0.9827953877537872 | pu |
| direct OpenDSS | 671 | 2 | 1.040273943965177 | pu |
| direct OpenDSS | 671 | 3 | 0.9648999396286697 | pu |


### Run the same feeder through CEPT

The learner-facing path is terminal-first. Colab uses !; a normal terminal uses the same command without it.

In [4]:
!cept study demo unbalanced-load-flow \
    --network ieee13 \
    --out runs/02-ieee13-unbalanced \
    --force \
    --format text

!cept study verify runs/02-ieee13-unbalanced --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the unbalanced load flow (IEEE 13-node) and saved the evidence
Saved run          runs\02-ieee13-unbalanced
Case fingerprint   4716c9c07f02 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\02-ieee13-unbalanced --format text


CEPT study check: PASSED
----------------------------
Study              Unbalanced load flow (OpenDSS)
Case fingerprint   4716c9c07f02 (matches the case you ran)

Checked   3 groups, 14 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (3 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     runs\02-ieee13-unbalanced\public-verification.json

For the full check list
  cept study verify runs\02-ieee13-unbalanced --format json


### Compare per phase

The optional detail cell reads the exact persisted results.json and checks bus 671 against the direct OpenDSS route. Keep the three phases separate; do not average away the imbalance.

In [5]:
#@title Compare solver outputs (optional details)
RUN_DIR = WORKSPACE / "runs" / "02-ieee13-unbalanced"
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
cept_rows = [row for row in results["load_flow"]["bus_voltages"] if row["bus"].lower() == BUS_TO_INSPECT.lower()]
cept_by_phase = {row["phase"]: row["v_pu"] for row in cept_rows}
table(
    ["phase", "direct OpenDSS pu", "CEPT pu", "|difference| pu"],
    [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)],
)
max_abs_diff_pu = max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3))
print()
print("Direct OpenDSS and CEPT agree on all 3 phases")
print("--------------------------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Largest phase difference: {max_abs_diff_pu:.2e} per unit (limit 1e-4 per unit)")
print("A small difference is rounding, not a different answer.")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max_abs_diff_pu < 1e-4


| phase | direct OpenDSS pu | CEPT pu | |difference| pu |
| --- | --- | --- | --- |
| 1 | 0.9827953877537872 | 0.982797 | 1.6122462128675963e-06 |
| 2 | 1.040273943965177 | 1.040275 | 1.0560348231436478e-06 |
| 3 | 0.9648999396286697 | 0.964889 | 1.0939628669714985e-05 |

Direct OpenDSS and CEPT agree on all 3 phases
--------------------------------------------
Result        PASSED
Largest phase difference: 1.09e-05 per unit (limit 1e-4 per unit)
A small difference is rounding, not a different answer.


The table is built from the direct solver readback and the CEPT run's persisted `results.json`. Do not average the phases to hide imbalance. The verified public workflow remains a bounded `WORKFLOW_VALIDATED` result and does not establish project or field validation.